In [ ]:
# Submission path setup: run notebooks from any submission subfolder.
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'Functions.ipynb').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Project root:', PROJECT_ROOT)


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)


In [ ]:
wide_csv_path = Path("early_search/evaluation_wide.csv")
wider_csv_path = Path("early_search/evaluation_wider.csv")

assert wide_csv_path.exists(), f"Could not find {wide_csv_path.resolve()}"
assert wider_csv_path.exists(), f"Could not find {wider_csv_path.resolve()}"

wide_df = pd.read_csv(wide_csv_path).assign(source="wide")
wider_df = pd.read_csv(wider_csv_path).assign(source="wider")

# The original sweep was the 50-epoch sweep even though that CSV does not store the epochs column.
if "epochs" not in wide_df.columns:
    wide_df["epochs"] = 50

df_raw = pd.concat([wide_df, wider_df], ignore_index=True, sort=False)
df = df_raw[df_raw["batch_size"].isin([1, 2, 5]) & (df_raw["epochs"] == 50)].copy()
excluded_df = df_raw[~(df_raw["batch_size"].isin([1, 2, 5]) & (df_raw["epochs"] == 50))].copy()

print("wide shape:", wide_df.shape)
print("wider shape:", wider_df.shape)
print("raw combined shape:", df_raw.shape)
print("filtered combined shape:", df.shape)
print("excluded rows:", len(excluded_df))
if len(excluded_df):
    display(excluded_df[[c for c in ["run_name", "source", "batch_size", "epochs"] if c in excluded_df.columns]].head(20))
display(df.head())


In [ ]:
print("columns:", list(df.columns))
print()
print("source counts:")
print(df["source"].value_counts())
print()
for col in ["arch_name", "loss_name", "batch_size", "lr", "weight_decay"]:
    if col in df.columns:
        print(f"{col}: {sorted(df[col].dropna().unique().tolist())}")


In [ ]:
top_k = 10

df = df.copy()
df["candidate_id"] = df["source"].astype(str) + "::" + df["run_name"].astype(str)

top_loss = df.nsmallest(top_k, "loss_val").copy()
top_dice = df.nlargest(top_k, "dice_val").copy()
top_hd95 = df.nsmallest(top_k, "hd95_val").copy()

print("Top 10 by validation loss")
display(top_loss[["source", "run_name", "arch_name", "loss_name", "lr", "weight_decay", "batch_size", "loss_val", "dice_val", "hd95_val"]])

print("Top 10 by validation Dice")
display(top_dice[["source", "run_name", "arch_name", "loss_name", "lr", "weight_decay", "batch_size", "loss_val", "dice_val", "hd95_val"]])

print("Top 10 by validation HD95")
display(top_hd95[["source", "run_name", "arch_name", "loss_name", "lr", "weight_decay", "batch_size", "loss_val", "dice_val", "hd95_val"]])

loss_ids = set(top_loss["candidate_id"])
dice_ids = set(top_dice["candidate_id"])
hd95_ids = set(top_hd95["candidate_id"])

print("Overlap counts")
print({
    "loss_only": len(loss_ids - dice_ids - hd95_ids),
    "dice_only": len(dice_ids - loss_ids - hd95_ids),
    "hd95_only": len(hd95_ids - loss_ids - dice_ids),
    "loss_and_dice": len((loss_ids & dice_ids) - hd95_ids),
    "loss_and_hd95": len((loss_ids & hd95_ids) - dice_ids),
    "dice_and_hd95": len((dice_ids & hd95_ids) - loss_ids),
    "all_three": len(loss_ids & dice_ids & hd95_ids),
    "total_unique": len(loss_ids | dice_ids | hd95_ids),
})

candidate_ids = loss_ids | dice_ids | hd95_ids
candidate_df = df[df["candidate_id"].isin(candidate_ids)].copy()
candidate_df["top10_loss_val"] = candidate_df["candidate_id"].isin(loss_ids)
candidate_df["top10_dice_val"] = candidate_df["candidate_id"].isin(dice_ids)
candidate_df["top10_hd95_val"] = candidate_df["candidate_id"].isin(hd95_ids)
candidate_df["selection_count"] = candidate_df[["top10_loss_val", "top10_dice_val", "top10_hd95_val"]].sum(axis=1)

# Lower validation loss is better, so this is sorted ascending by loss_val.
candidate_df = candidate_df.sort_values(["loss_val", "dice_val", "hd95_val"], ascending=[True, False, True])

display_cols = [
    "source", "run_name", "arch_name", "loss_name", "channels", "strides", "lr", "weight_decay", "batch_size",
    "top10_loss_val", "top10_dice_val", "top10_hd95_val", "selection_count",
    "loss_train", "loss_val", "loss_test",
    "dice_train", "dice_val", "dice_test",
    "hd95_train", "hd95_val", "hd95_test",
]

print("Combined shortlist from the three top-10 selections")
display(candidate_df[display_cols].reset_index(drop=True))


In [ ]:
print("Selection count distribution")
display(candidate_df["selection_count"].value_counts().sort_index().rename("runs").to_frame())

for col in ["source", "arch_name", "loss_name", "batch_size", "lr", "weight_decay"]:
    print(f"Spread across {col}")
    display(candidate_df[col].value_counts().sort_index().rename("runs").to_frame())

print("Architecture by selection count")
display(pd.crosstab(candidate_df["arch_name"], candidate_df["selection_count"]))

print("Source by selection count")
display(pd.crosstab(candidate_df["source"], candidate_df["selection_count"]))


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

plot_cols = ["source", "arch_name", "loss_name", "batch_size", "lr", "weight_decay"]

for ax, col in zip(axes.ravel(), plot_cols):
    counts = candidate_df[col].value_counts().sort_index()
    ax.bar(counts.index.astype(str), counts.values)
    ax.set_title(col)
    ax.set_ylabel("runs")
    ax.tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()

selection_counts = candidate_df["selection_count"].value_counts().sort_index()
plt.figure(figsize=(5, 3.5))
plt.bar(selection_counts.index.astype(str), selection_counts.values)
plt.title("How many top-10 lists selected each run")
plt.xlabel("selection_count")
plt.ylabel("runs")
plt.show()
